# Explore a reply tree

Run the cells, enter a Reddit post ID such as `t3_18dj9h`, and choose a top-level reply. Lines show direct parent–child relationships. Teal lines show replies between the OP and the opening challenger. **Stars mark delta recipients** and **diamonds mark the OP comments awarding those deltas**. Click a node to read the archived comment and see its parent, depth, and time since the post.

If the widget imports fail, run `%pip install plotly ipywidgets anywidget` in this notebook's kernel.

In [ ]:
from collections import defaultdict
from contextlib import closing
from html import escape
from pathlib import Path
import re

import ipywidgets as widgets
import plotly.graph_objects as go
from IPython.display import display
from psycopg import Error as DatabaseError
from psycopg.rows import dict_row

from cmv_data import Database, TextStore


REPO_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "team-setup").is_dir() and (path / "pyproject.toml").is_file()
)
DEFAULT_POST_ID = "t3_18dj9h"
DEFAULT_BRANCH_ID = "t1_c8dtv3e"
CONFIRMATION = re.compile(r"^confirmed\s*[-–—]\s*\d+\s+deltas?\s+awarded\s+to\b", re.I)
store = TextStore(REPO_ROOT / "project/data/processed/cmv-text/v1", expected_version="v1")

## Load the discussion and verify awards

PostgreSQL gives us parent links, authors, and timestamps. Parquet provides the post and comment bodies. A delta is highlighted when an archived DeltaBot confirmation links to an award written by the OP and to its recipient.

In [ ]:
def author_key(row):
    return (row["author"] or "").casefold()


def load_discussion(post_id):
    """Load one archived discussion. Keep IDs and text local to the kernel."""
    if not re.fullmatch(r"t3_[a-z0-9]+", post_id):
        raise ValueError("Enter a Reddit post ID such as t3_18dj9h.")

    with closing(Database(env_file=REPO_ROOT / ".env")) as db:
        with db.connection.transaction(), db.connection.cursor(row_factory=dict_row) as cursor:
            cursor.execute("SET TRANSACTION READ ONLY")
            cursor.execute(
                "SELECT post_id AS node_id, author, title, created_utc FROM cmv.posts WHERE post_id = %s",
                (post_id,),
            )
            post = cursor.fetchone()
            if post is None:
                raise ValueError(f"No archived post was found for {post_id}.")
            cursor.execute("""
                SELECT comment_id AS node_id, author, created_utc,
                       COALESCE(parent_comment_id, original_parent_id) AS parent_id
                FROM cmv.comments
                WHERE post_id = %s
                ORDER BY created_utc NULLS LAST, comment_id
            """, (post_id,))
            comments = cursor.fetchall()

    rows = {post_id: {**post, "parent_id": None}}
    rows.update({row["node_id"]: row for row in comments})
    children = defaultdict(list)
    for row in comments:
        children[row["parent_id"]].append(row["node_id"])

    # A valid database post may have no text in the pinned Parquet release.
    try:
        bodies = {row["comment_id"]: row["body"] for row in store.get_comments(post_id)}
        bodies[post_id] = store.get_post(post_id)["body"]
    except (KeyError, FileNotFoundError):
        bodies = {}

    # DeltaBot reply -> OP award comment -> comment that received the delta.
    awards = {}
    op = author_key(post)
    if op:
        for row in comments:
            if author_key(row) != "deltabot":
                continue
            body = (bodies.get(row["node_id"]) or "").strip().lstrip("*_").strip()
            award = rows.get(row["parent_id"])
            if not CONFIRMATION.match(body) or not award or author_key(award) != op:
                continue
            recipient = award["parent_id"]
            if recipient in rows and recipient != post_id:
                awards[award["node_id"]] = recipient

    return {
        "post_id": post_id, "post": post, "rows": rows, "children": children,
        "bodies": bodies, "awards": awards,
    }

## Draw the selected branch

The graph keeps the original post visible and follows one opening's descendants. Horizontal position shows depth. Vertical position separates branches of the same parent. Hover to preview a comment and click a node to read the full text.

In [ ]:
def branch_layout(post_id, opening_id, children):
    """Lay out the selected branch and its original post in parent-to-child order."""
    depths = {}
    edges = []
    pending = [(post_id, -1, None)]
    while pending:
        node_id, depth, parent_id = pending.pop()
        if node_id in depths:
            continue
        depths[node_id] = depth
        if parent_id is not None:
            edges.append((parent_id, node_id))
        replies = [opening_id] if node_id == post_id else children.get(node_id, ())
        pending.extend((child, depth + 1, node_id) for child in reversed(replies))

    # Give each leaf its own row. Place each parent midway between its children.
    visible_children = defaultdict(list)
    for parent_id, child_id in edges:
        visible_children[parent_id].append(child_id)
    y = {node_id: index for index, node_id in enumerate(
        node_id for node_id in depths if not visible_children[node_id]
    )}
    for node_id in reversed(depths):
        replies = visible_children[node_id]
        if replies:
            y[node_id] = (y[replies[0]] + y[replies[-1]]) / 2
    return depths, edges, y


def branch_figure(discussion, opening_id, select_node):
    post_id = discussion["post_id"]
    rows = discussion["rows"]
    bodies = discussion["bodies"]
    awards = discussion["awards"]
    recipients = set(awards.values())
    depths, edges, y = branch_layout(post_id, opening_id, discussion["children"])
    op = author_key(discussion["post"])
    challenger = author_key(rows[opening_id])

    def role(node_id):
        name = author_key(rows[node_id])
        if node_id == post_id:
            return "Original post"
        if name and name == op:
            return "OP reply"
        if node_id == opening_id or (name and name == challenger):
            return "Opening challenger"
        if name == "deltabot":
            return "DeltaBot"
        return "Other commenter"

    # Store both endpoints for every edge. Color direct OP–challenger exchanges.
    edge_points = defaultdict(lambda: ([], []))
    for parent_id, child_id in edges:
        is_exchange = {role(parent_id), role(child_id)} == {"OP reply", "Opening challenger"}
        x_points, y_points = edge_points[is_exchange]
        x_points.extend((depths[parent_id], depths[child_id], None))
        y_points.extend((y[parent_id], y[child_id], None))

    figure = go.FigureWidget()
    for is_exchange, (x_points, y_points) in edge_points.items():
        figure.add_scatter(
            x=x_points, y=y_points, mode="lines", hoverinfo="skip",
            name="OP–challenger exchange" if is_exchange else "Other parent–child link",
            line={"color": "#16828a" if is_exchange else "#aab5c0",
                  "width": 2.5 if is_exchange else 1.25},
        )

    marks = {
        "Original post": ("square", "#334155", 17),
        "Opening challenger": ("triangle-right", "#128071", 15),
        "OP reply": ("circle", "#2b62ad", 14),
        "Other commenter": ("circle-open", "#60738a", 12),
        "DeltaBot": ("x", "#7d8b99", 10),
        "Delta recipient": ("star", "#bd315a", 22),
        "OP delta award": ("diamond", "#c05b1b", 18),
    }
    groups = defaultdict(list)
    for node_id in depths:
        kind = ("OP delta award" if node_id in awards
                else "Delta recipient" if node_id in recipients else role(node_id))
        groups[kind].append(node_id)

    labels = {post_id: "OP post", opening_id: "Opening"}
    labels.update({node_id: "Delta award" for node_id in awards})
    labels.update({node_id: "Received delta" for node_id in recipients})
    if opening_id in recipients:
        labels[opening_id] = "Opening received delta"

    def on_click(trace, points, _state):
        if points.point_inds:
            select_node(trace.customdata[points.point_inds[0]])

    for kind, (symbol, color, size) in marks.items():
        ids = groups[kind]
        if not ids:
            continue
        hover = [
            f"<b>{escape(role(node_id))}</b> · {escape(rows[node_id]['author'] or 'Unknown')}<br>"
            f"{node_id} · {'Original post' if node_id == post_id else f'Depth {depths[node_id]}'}<br>"
            f"{escape(' '.join((bodies.get(node_id) or '').split())[:120])}"
            for node_id in ids
        ]
        figure.add_scatter(
            x=[depths[node_id] for node_id in ids], y=[y[node_id] for node_id in ids],
            mode="markers+text", name=kind, customdata=ids, hovertext=hover,
            hovertemplate="%{hovertext}<extra></extra>", textposition="top center",
            text=[labels.get(node_id, "") for node_id in ids],
            marker={"symbol": symbol, "color": color, "size": size, "line": {"width": 1}},
        )
        figure.data[-1].on_click(on_click)

    max_depth = max(depths.values())
    figure.update_layout(
        template="plotly_white", height=min(780, max(330, 250 + 20 * (1 + max(y.values())))),
        margin={"l": 24, "r": 28, "t": 30, "b": 82}, hovermode="closest",
        legend={"orientation": "h", "y": -0.2, "x": 0},
        xaxis={"title": "Reply depth", "range": [-1.6, max_depth + 0.6],
               "tickmode": "array", "tickvals": list(range(-1, max_depth + 1)),
               "ticktext": ["Post", *map(str, range(max_depth + 1))], "zeroline": False},
        yaxis={"visible": False, "range": [max(y.values()) + 1, -1]},
    )
    return figure, depths, role

## Choose a post and a branch

The dropdown lists direct replies to the loaded post. Changing the selection redraws the tree. The detail panel shows the entire selected comment, and identifies the recipient or the OP award when either was found in the archive.

In [ ]:
post_input = widgets.Text(value=DEFAULT_POST_ID, description="Post ID:")
load_button = widgets.Button(description="Load post", button_style="primary")
branch_picker = widgets.Dropdown(description="Opening:", layout=widgets.Layout(width="100%"))
node_picker = widgets.Dropdown(description="Comment:", layout=widgets.Layout(width="100%"))
status = widgets.HTML()
graph_area = widgets.VBox()
detail = widgets.HTML()
state = {"discussion": None, "depths": {}, "role": None}


def show_node(change):
    node_id = change["new"]
    discussion = state["discussion"]
    if not node_id or not discussion:
        detail.value = ""
        return
    row = discussion["rows"][node_id]
    post_time = discussion["post"]["created_utc"]
    metadata = []
    if node_id != discussion["post_id"]:
        metadata.extend([f"Depth {state['depths'][node_id]}", f"Parent {row['parent_id']}"])
    if row["created_utc"] is not None and post_time is not None:
        metadata.append(f"{(row['created_utc'] - post_time) / 60:.1f} min after post")
    awards = discussion["awards"]
    if node_id in awards:
        metadata.append(f"Awarded a delta to {awards[node_id]}")
    if node_id in awards.values():
        metadata.append("Received an OP-awarded delta")

    body = discussion["bodies"].get(node_id) or "Archived text unavailable."
    detail.value = (
        f"<h3>{escape(state['role'](node_id))} · {escape(row['author'] or 'Unknown')}</h3>"
        f"<p><code>{escape(node_id)}</code> · {escape(' · '.join(metadata))}</p>"
        f'<div style="white-space:pre-wrap;overflow-wrap:anywhere">{escape(body)}</div>'
    )


def show_branch(change):
    opening_id = change["new"]
    for old_figure in graph_area.children:
        old_figure.close()
    graph_area.children = ()
    node_picker.options = [("Select a comment", None)]
    if not opening_id:
        return

    discussion = state["discussion"]
    figure, depths, role = branch_figure(
        discussion, opening_id, lambda node_id: setattr(node_picker, "value", node_id)
    )
    state.update(depths=depths, role=role)
    node_picker.options = [
        (f"{role(node_id)} · {discussion['rows'][node_id]['author'] or 'Unknown'} · {node_id}", node_id)
        for node_id in depths
    ]
    node_picker.value = opening_id
    graph_area.children = (figure,)


def on_load(_button):
    post_id = post_input.value.strip().lower()
    try:
        discussion = load_discussion(post_id)
    except ValueError as error:
        status.value = f"<b>Could not load post.</b> {escape(str(error))}"
        return
    except DatabaseError:
        status.value = "<b>Could not connect to the database.</b> Check the project's database connection."
        return

    # Clear the previous selection before replacing the post and its available branches.
    branch_picker.options = [("Select an opening", None)]
    state["discussion"] = discussion
    openings = discussion["children"][post_id]
    options = []
    for node_id in openings:
        row = discussion["rows"][node_id]
        preview = " ".join((discussion["bodies"].get(node_id) or "").split())[:54]
        options.append((f"{row['author'] or 'Unknown'} · {node_id} · {preview}", node_id))
    title = discussion["post"]["title"] or post_id
    status.value = (
        f"<b>{escape(title)}</b><br>{len(openings)} top-level replies · "
        f"{len(discussion['awards'])} confirmed OP delta awards in the archive"
    )
    if options:
        branch_picker.options = options
        branch_picker.value = DEFAULT_BRANCH_ID if DEFAULT_BRANCH_ID in openings else openings[0]
    else:
        # A post with zero direct replies still has readable content.
        state.update(depths={post_id: -1}, role=lambda _node: "Original post")
        node_picker.options = [("Original post", post_id)]


branch_picker.observe(show_branch, names="value")
node_picker.observe(show_node, names="value")
load_button.on_click(on_load)
on_load(load_button)
display(widgets.VBox([
    widgets.HBox([post_input, load_button]), status, branch_picker,
    graph_area, node_picker, detail,
]))